# AdS4 Phase-Lock Probe-Break Test (v8)

This notebook tests when constraint stacking fails.

We:
- keep architecture fixed
- weaken or remove one probe
- observe error growth and degeneracy return


In [ ]:
import torch, math
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt


In [ ]:
# simple ground truth
r = torch.linspace(1.05,3.5,200).view(-1,1)
def f_true(r): return r**2 + 1 - 1/r
def g_true(r): return 1/(f_true(r)+1e-6)


In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1,64), nn.Tanh(),
            nn.Linear(64,64), nn.Tanh(),
            nn.Linear(64,1)
        )
    def forward(self,x): return F.softplus(self.net(x))

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = MLP()
        self.g = MLP()
    def forward(self,r):
        return self.f(r), self.g(r)


In [ ]:
def train(use_geo=True, noise=0.0):
    m = Model()
    opt = torch.optim.Adam(m.parameters(), lr=2e-3)
    loss_hist = []
    for _ in range(800):
        opt.zero_grad()
        f,g = m(r)

        loss = ((f - f_true(r))**2).mean()
        loss += ((g - g_true(r))**2).mean()

        if use_geo:
            loss += ((torch.sqrt(f+g) - torch.sqrt(f_true(r)+g_true(r)))**2).mean()

        # corruption
        loss += noise * torch.rand(1)

        loss.backward()
        opt.step()
        loss_hist.append(loss.item())

    return m, loss_hist


In [ ]:
# full stack
m_full, l_full = train(use_geo=True, noise=0.0)

# remove GEO
m_drop, l_drop = train(use_geo=False, noise=0.0)

# corrupt GEO
m_noise, l_noise = train(use_geo=True, noise=0.01)


In [ ]:
plt.plot(l_full, label='full stack')
plt.plot(l_drop, label='no GEO')
plt.plot(l_noise, label='corrupt GEO')
plt.legend(); plt.title('probe-break loss'); plt.show()

In [ ]:
f_full,g_full = m_full(r)
f_drop,g_drop = m_drop(r)

plt.plot(r, f_true(r), label='true')
plt.plot(r, f_full, '--', label='full')
plt.plot(r, f_drop, ':', label='drop GEO')
plt.legend(); plt.title('degeneracy return'); plt.show()